# Inspect event and audience labels

Label analysis identifies rare argument types, time-role imbalance, and gaps in applicability examples. Use validated canonical annotations so every count refers to a recoverable span or event.

The default dataset is synthetic. Its plots check the workflow and do not support research conclusions. Change `DATASET` in the setup cell after collecting and preparing a reviewed corpus.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from smartnotice.demo import run_demo
from smartnotice.io import read_jsonl
from smartnotice.schemas import Notice

%matplotlib inline

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), (
    "Open this notebook from the repository root or notebooks/"
)

DATASET = "demo"  # Set to "research" to read the prepared local corpus.
if DATASET == "demo":
    run_demo(ROOT)
    NOTICE_PATH = ROOT / "data/interim/demo/notices.jsonl"
    SPLIT_PATH = ROOT / "data/processed/demo/splits.jsonl"
    ANNOTATION_PATH = ROOT / "data/annotations/examples/events.jsonl"
    PAIR_PATH = ROOT / "data/processed/demo/applicability.jsonl"
else:
    NOTICE_PATH = ROOT / "data/interim/notices.jsonl"
    SPLIT_PATH = ROOT / "data/processed/splits.jsonl"
    ANNOTATION_PATH = ROOT / "data/annotations/adjudicated.jsonl"
    PAIR_PATH = ROOT / "data/processed/applicability.jsonl"

FIGURES = ROOT / "reports/generated" / DATASET / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
notices = [Notice.model_validate(row) for row in read_jsonl(NOTICE_PATH)]
assert notices, "Prepare at least one notice before analysis"
frame = pd.DataFrame([n.model_dump(mode="json") for n in notices])
print(f"Dataset: {DATASET}; documents: {len(notices)}; synthetic: {int(frame.synthetic.sum())}")
if DATASET == "research":
    assert not frame.synthetic.any(), "Exclude synthetic fixtures from research measurements"

In [ ]:
from smartnotice.labeling.analysis import annotation_summary
from smartnotice.labeling.validation import validate_annotation
from smartnotice.schemas import Annotation

annotations = [Annotation.model_validate(row) for row in read_jsonl(ANNOTATION_PATH)]
documents = {notice.doc_id: notice for notice in notices}
for annotation in annotations:
    validate_annotation(annotation, documents[annotation.doc_id])
summary = annotation_summary(annotations)
span_counts = pd.Series(summary["span_counts"], dtype="int64").reindex(
    ["ACTION", "TIME", "AUDIENCE", "CONDITION", "CHANNEL", "DOCUMENT"], fill_value=0
)
role_counts = pd.Series(summary["time_role_counts"], dtype="int64").reindex(
    ["START", "DEADLINE", "EVENT_TIME"], fill_value=0
)
display(span_counts.rename("spans").to_frame())
display(role_counts.rename("time links").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
span_counts.plot.bar(ax=axes[0], title="Argument and action spans", rot=30)
role_counts.plot.bar(ax=axes[1], title="Time roles", rot=20)
for axis in axes:
    axis.set_ylabel("Count")
fig.tight_layout()
fig.savefig(FIGURES / "labels_span_role_counts.png", dpi=160)
plt.show()
plt.close("all")

In [ ]:
distances = pd.Series(summary["argument_distances_chars"], name="distance_chars", dtype="float64")
display(distances.describe().to_frame())
if not distances.empty:
    axis = distances.plot.hist(
        bins=min(12, len(distances)), title="Action-to-argument distance", figsize=(8, 3.5)
    )
    axis.set_xlabel("Absolute distance between span starts (characters)")
    axis.figure.tight_layout()
    axis.figure.savefig(FIGURES / "labels_argument_distance.png", dpi=160)
    plt.show()
plt.close("all")

## Inspect silver applicability labels

These labels come from reviewed constraints and synthetic profiles. They measure the output of the labeling rules, not model predictions or independent human agreement. Inherit splits from each document before training.

In [ ]:
pairs = pd.DataFrame(read_jsonl(PAIR_PATH))
if not pairs.empty:
    assigned = pd.DataFrame(read_jsonl(SPLIT_PATH))
    pair_splits = pairs.merge(
        assigned[["doc_id", "split"]], on="doc_id", how="left", validate="many_to_one"
    )
    assert pair_splits.split.notna().all()
    display(
        pd.crosstab(pair_splits["split"], pair_splits["label"]).reindex(
            columns=["APPLIES", "NOT_APPLIES", "UNCLEAR"], fill_value=0
        )
    )
    display(pairs.groupby(["label", "rule_reason"]).size().rename("pairs").to_frame())
    label_counts = pairs.label.value_counts().reindex(
        ["APPLIES", "NOT_APPLIES", "UNCLEAR"], fill_value=0
    )
    axis = label_counts.plot.bar(
        rot=15, title="Silver applicability label counts", figsize=(7, 3.5)
    )
    axis.set_ylabel("Event/profile pairs")
    axis.figure.tight_layout()
    axis.figure.savefig(FIGURES / "labels_applicability.png", dpi=160)
    plt.show()
plt.close("all")

## Measure independent agreement

No independent human annotation is included in the demo. Follow `docs/annotation.md` to align decisions before using `span_f1` and `cohens_kappa`. Record the number of documents and aligned decisions, exact matching policy, annotator pairing, and adjudication outcomes. A self-comparison of a synthetic annotation is not an agreement result.

In [ ]:
counts_per_document = pd.Series([annotation.doc_id for annotation in annotations]).value_counts()
print("Documents with multiple annotation rows:", int((counts_per_document > 1).sum()))
print(
    "Canonical adjudicated labels and independent pre-adjudication labels serve different analyses."
)